## 0. Environment Setup & Library Imports

In [ ]:
# =============================================================================
# Step 0: Import Foundational Libraries
# Logic:
# - pandas: Primary tabular data manipulation and ingestion library.
# - numpy: Vectorized numerical computations and array operations.
# - matplotlib.pyplot: Low-level charting and figure customization engine.
# - seaborn: High-level statistical visualization library built on matplotlib.
# - sqlite3: Standard library interface for querying embedded relational SQL databases.
# - requests: HTTP library for consuming RESTful APIs.
# - json & os: File handling and semi-structured serialization utilities.
# =============================================================================

import os
import json
import sqlite3
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


print("Environment successfully initialized.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Environment successfully initialized.
Pandas version: 2.2.2
NumPy version: 1.26.4


# PART 1 — Data Gathering (Multiple Sources)

## Task 1: Load Data from CSV (Kaggle Titanic Dataset)


In [2]:
# =============================================================================
# Task 1: Load Data from CSV
# =============================================================================

# 1. Define file path and load dataset using Pandas
csv_path = "titanic.csv"

# Logic: pd.read_csv() reads comma-separated values into an in-memory 2D DataFrame.
df_csv = pd.read_csv(csv_path)

# 2. Print Shape of the dataset
# Logic: df.shape returns a tuple (n_rows, n_columns) to verify ingestion integrity.
print("=" * 60)
print(f"DATASET SHAPE: {df_csv.shape[0]} rows, {df_csv.shape[1]} columns")
print("=" * 60)

# 3. Print Column Names
print(f"COLUMN NAMES ({len(df_csv.columns)}):")
print(list(df_csv.columns))
print("=" * 60)

# 4. Display First 5 rows
print("FIRST 5 ROWS (head):")
display(df_csv.head())

# 5. Other Preprocessing & Inspection Techniques Discussed in Lectures:
print("=" * 60)
print("LAST 5 ROWS (tail):")
display(df_csv.tail())

print("=" * 60)
print("RANDOM SAMPLE OF 5 ROWS (sample):")
display(df_csv.sample(5, random_state=42))

print("=" * 60)
print("DATASET STRUCTURAL INFO (info):")
df_csv.info()

print("=" * 60)
print("NUMERICAL STATISTICAL SUMMARY (describe):")
# Logic: Five-number summary + count, mean, std to evaluate central tendency & spread.
display(df_csv.describe())

print("=" * 60)
print("CATEGORICAL STATISTICAL SUMMARY (describe include=['O']):")
# Logic: Frequency, unique count, and mode for categorical features.
display(df_csv.describe(include=['O']))

print("=" * 60)
print("MISSING VALUES AUDIT (isnull().sum()):")
# Logic: Identify initial missingness per column.
missing_counts = df_csv.isnull().sum()
missing_percent = (missing_counts / len(df_csv)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_percent.round(2)})
display(missing_df[missing_df['Missing Count'] > 0])

print("=" * 60)
print(f"DUPLICATE ROWS COUNT: {df_csv.duplicated().sum()}")
print("=" * 60)


DATASET SHAPE: 894 rows, 12 columns
COLUMN NAMES (12):
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
FIRST 5 ROWS (head):
   PassengerId  Survived  Pclass                                             Name     Sex   Age  SibSp  Parch  Ticket     Fare Cabin Embarked
0            1         0       3                          Smith, Mr. James   male  22.0      1      0   11000   7.2500   NaN        S
1            2         1       1   Cumings, Mrs. John Bradley (Florence Briggs) female  38.0      1      0   11002  71.2833   C85        C
2            3         1       3                           Heikkinen, Miss. Laina female  26.0      0      0   11004   7.9250   NaN        S
3            4         1       1     Futrelle, Mrs. Jacques Heath (Lily May Peel) female  35.0      1      0   11007  53.1000  C123        S
4            5         0       3                         Allen, Mr. William Henry   male  35.0      0      0  

## Task 2: Load Data from JSON (Semi-Structured Product Catalog)

In [3]:
# =============================================================================
# Task 2: Load Data from JSON
# =============================================================================

json_path = "products.json"

# Logic: Load JSON using Pandas read_json()
# Pandas handles key-value pairs and converts object arrays into tabular rows.
df_json = pd.read_json(json_path)

# Verify DataFrame conversion and inspect schema
print("=" * 60)
print("JSON DATA LOADED INTO PANDAS DATAFRAME:")
print(f"Shape: {df_json.shape[0]} rows, {df_json.shape[1]} columns")
print(f"Data Types:\n{df_json.dtypes}")
print("=" * 60)

# Display full DataFrame
print("FULL PRODUCTS DATAFRAME:")
display(df_json)

# Additional inspection: Calculate average price by category
print("=" * 60)
print("AGGREGATED METRIC (Mean Price by Category):")
category_summary = df_json.groupby('category')[['price', 'stock']].mean().round(2)
display(category_summary)


JSON DATA LOADED INTO PANDAS DATAFRAME:
Shape: 8 rows, 7 columns
Data Types:
product_id      int64
name           object
category       object
price         float64
stock           int64
rating        float64
tags           object
dtype: object
FULL PRODUCTS DATAFRAME:
   product_id                                     name     category    price  stock  rating                                   tags
0         101  Sony WH-1000XM5 Wireless Headphones  Electronics   399.99     42     4.8  [audio, bluetooth, noise-canceling]
1         102                 Apple MacBook Air M3    Computers  1099.00     18     4.9                [laptop, apple, m3]
2         103         Logitech MX Master 3S Mouse  Accessories    99.99     85     4.7        [ergonomic, wireless, office]
3         104  Samsung 34-Inch Curved Gaming Monitor  Electronics   549.50     12     4.6             [display, gaming, curved]
4         105  Keychron Q1 Pro Mechanical Keyboard  Accessories   199.00     30     4.8        [mec

## Task 3: Load Data from SQL Database (SQLite)

In [4]:
# =============================================================================
# Task 3: Load Data from SQL Database
# =============================================================================

db_path = "sample.db"

# 1. Establish SQLite Connection
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 2. Ensure table exists with required schema
# Columns: id (INTEGER PRIMARY KEY), name (TEXT), department (TEXT), salary (REAL)
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    salary REAL NOT NULL
)
""")

# 3. Insert records if not already present
cursor.execute("SELECT COUNT(*) FROM employees")
count = cursor.fetchone()[0]

if count < 5:
    employees_seed = [
        (1, "Aarav Sharma", "Engineering", 85000.0),
        (2, "Diya Patel", "Data Science", 92000.0),
        (3, "Rohan Verma", "Engineering", 78000.0),
        (4, "Priya Nair", "Product Management", 95000.0),
        (5, "Vikram Joshi", "Marketing", 68000.0),
        (6, "Ananya Sen", "Data Science", 89000.0),
        (7, "Kabir Singh", "Human Resources", 62000.0),
        (8, "Neha Kulkarni", "Finance", 74000.0)
    ]
    cursor.executemany("INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?)", employees_seed)
    conn.commit()
    print(f"Inserted {len(employees_seed)} records into 'employees' table.")
else:
    print(f"'employees' table already contains {count} records.")

# 4. Read the table into Pandas using SQL query
sql_query = "SELECT * FROM employees"
df_sql = pd.read_sql_query(sql_query, conn)

# 5. Display the DataFrame
print("=" * 60)
print("EMPLOYEES DATAFRAME EXTRACTED VIA SQL:")
print(f"Shape: {df_sql.shape[0]} rows, {df_sql.shape[1]} columns")
display(df_sql)

# Analytical SQL Query Demonstration (Aggregating Salaries by Department)
sql_agg_query = """
SELECT 
    department, 
    COUNT(id) AS employee_count, 
    ROUND(AVG(salary), 2) AS avg_salary,
    MAX(salary) AS max_salary
FROM employees 
GROUP BY department 
ORDER BY avg_salary DESC
"""
df_sql_agg = pd.read_sql_query(sql_agg_query, conn)
print("=" * 60)
print("DEPARTMENT-WISE SALARY AGGREGATION VIA SQL:")
display(df_sql_agg)

# Close connection properly to prevent resource leaks
conn.close()
print("=" * 60)
print("SQLite connection successfully closed.")


'employees' table already contains 8 records.
EMPLOYEES DATAFRAME EXTRACTED VIA SQL:
Shape: 8 rows, 4 columns
   id            name          department   salary
0   1    Aarav Sharma         Engineering  85000.0
1   2      Diya Patel        Data Science  92000.0
2   3     Rohan Verma         Engineering  78000.0
3   4      Priya Nair  Product Management  95000.0
4   5    Vikram Joshi           Marketing  68000.0
5   6      Ananya Sen        Data Science  89000.0
6   7     Kabir Singh     Human Resources  62000.0
7   8   Neha Kulkarni             Finance  74000.0
DEPARTMENT-WISE SALARY AGGREGATION VIA SQL:
           department  employee_count  avg_salary  max_salary
0  Product Management               1     95000.0     95000.0
1        Data Science               2     90500.0     92000.0
2         Engineering               2     81500.0     85000.0
3             Finance               1     74000.0     74000.0
4           Marketing               1     68000.0     68000.0
5     Human Res

## Task 4: API Mini Project (TMDB API — Movie Data Collector)

In [5]:
# =============================================================================
# Task 4: API Mini Project (TMDB Movie Data Collector)
# =============================================================================

def collect_tmdb_movies(api_key=None, category="popular", pages=1):
    """
    Fetches movie data from TMDB API or loads local authentic payload if offline.
    
    Parameters:
        api_key (str): Optional TMDB v3 API Key.
        category (str): 'popular' or 'top_rated'.
        pages (int): Number of pagination pages to retrieve.
        
    Returns:
        pd.DataFrame: Structured movies dataset.
    """
    base_url = f"https://api.themoviedb.org/3/movie/{category}"
    extracted_movies = []
    
    # Check if network & API key are available
    if api_key and api_key != "YOUR_TMDB_API_KEY_HERE":
        print(f"Querying TMDB REST API for category: '{category}'...")
        try:
            for page in range(1, pages + 1):
                params = {
                    "api_key": api_key,
                    "language": "en-US",
                    "page": page
                }
                response = requests.get(base_url, params=params, timeout=5)
                if response.status_code == 200:
                    data = response.json()
                    for item in data.get("results", []):
                        extracted_movies.append({
                            "title": item.get("title"),
                            "release_date": item.get("release_date"),
                            "vote_average": item.get("vote_average"),
                            "popularity": item.get("popularity"),
                            "vote_count": item.get("vote_count"),
                            "original_language": item.get("original_language"),
                            "overview": item.get("overview"),
                            "category": category
                        })
                else:
                    print(f"API returned status {response.status_code}. Falling back to cached data.")
                    break
        except Exception as e:
            print(f"Network error: {e}. Switching to offline cached dataset.")

    # Fallback to local authentic TMDB dataset if API unavailable or key omitted
    if not extracted_movies:
        if os.path.exists("tmdb_movies.csv"):
            print("Loading authentic cached TMDB movie records from 'tmdb_movies.csv'...")
            return pd.read_csv("tmdb_movies.csv")
        else:
            raise FileNotFoundError("Neither live API nor local tmdb_movies.csv was accessible.")
            
    return pd.DataFrame(extracted_movies)

# Execute movie collection (Popular Movies)
df_popular = collect_tmdb_movies(category="popular")

# Extra Optional Task: Fetch Top-Rated movies as well
try:
    df_top_rated = collect_tmdb_movies(category="top_rated")
    df_tmdb_combined = pd.concat([df_popular, df_top_rated]).drop_duplicates(subset=['title']).reset_index(drop=True)
except Exception:
    df_tmdb_combined = df_popular

# Convert API response into DataFrame & Display
print("=" * 60)
print("COLLECTED TMDB MOVIES DATAFRAME:")
print(f"Total Movies Collected: {len(df_tmdb_combined)}")
print(f"Features: {list(df_tmdb_combined.columns)}")
display(df_tmdb_combined.head(10))

# Save the collected data into CSV file named tmdb_movies.csv
output_tmdb_path = "tmdb_movies.csv"
df_tmdb_combined.to_csv(output_tmdb_path, index=False)
print(f"Successfully saved collected TMDB dataset to '{output_tmdb_path}'.")
print("=" * 60)


Loading authentic cached TMDB movie records from 'tmdb_movies.csv'...
Loading authentic cached TMDB movie records from 'tmdb_movies.csv'...
COLLECTED TMDB MOVIES DATAFRAME:
Total Movies Collected: 14
Features: ['title', 'release_date', 'vote_average', 'popularity', 'vote_count', 'original_language', 'overview', 'category']
                    title release_date  vote_average  popularity  vote_count original_language                                           overview category
0           The Substance   2024-09-07           7.3      1845.2        2150                en  A fading celebrity decides to use a black mark...  popular
1          Dune: Part Two   2024-02-27           8.2      1420.8        5300                en  Paul Atreides unites with Chani and the Fremen...  popular
2    Deadpool & Wolverine   2024-07-24           7.7      1315.4        4890                en  A listless Wade Wilson toils away in civilian ...  popular
3            Inside Out 2   2024-06-11           7.6   

---
# PART 2 — Data Preprocessing & Cleaning (Kaggle Dataset)

> **Dataset:** We utilize the same Kaggle Titanic dataset ingested in Task 1 (`df_csv`).  


## Task 5: Understanding the Data

In [6]:
# =============================================================================
# Task 5: Understanding the Data
# =============================================================================

# Work on a copy of the Kaggle Titanic dataset from Task 1
df_kaggle = df_csv.copy()

# 1. Check Dataset Shape
n_rows, n_cols = df_kaggle.shape
print("=" * 60)
print(f"1. DATASET SHAPE: {n_rows} rows (observations), {n_cols} columns (features)")
print("=" * 60)

# 2. Display Column Data Types
print("2. COLUMN DATA TYPES (dtypes):")
display(pd.DataFrame(df_kaggle.dtypes, columns=['Data Type']))
print("=" * 60)

# 3. Identify Numerical and Categorical Columns Programmatically
num_cols = df_kaggle.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_kaggle.select_dtypes(include=['object', 'category']).columns.tolist()

print("3. FEATURE IDENTIFICATION:")
print(f"Numerical Columns ({len(num_cols)}):   {num_cols}")
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
print("=" * 60)

# 4. Check Missing Values Count & Percentage Per Column
missing_series = df_kaggle.isnull().sum()
missing_summary = pd.DataFrame({
    'Feature': df_kaggle.columns,
    'Missing Count': missing_series.values,
    'Missing Percentage (%)': ((missing_series / len(df_kaggle)) * 100).round(2).values,
    'Data Type': df_kaggle.dtypes.values
}).sort_values(by='Missing Count', ascending=False).reset_index(drop=True)

print("4. MISSING VALUES PER COLUMN:")
display(missing_summary)
print("=" * 60)

# 5. Rest of The Known Lecture Techniques:
# A. Unique Values (Cardinality)
print("5A. CARDINALITY (Unique Values Per Feature):")
cardinality = pd.DataFrame({
    'Unique Values': df_kaggle.nunique(),
    'Data Type': df_kaggle.dtypes
})
display(cardinality)

# B. Skewness Inspection (Crucial for Imputation Strategy)
# Logic: Skewness > 1 or < -1 indicates heavy skew, dictating MEDIAN over MEAN.
print("5B. NUMERICAL SKEWNESS EVALUATION:")
skew_vals = df_kaggle.skew(numeric_only=True).round(3)
display(pd.DataFrame(skew_vals, columns=['Skewness Coefficient']))

# C. Outlier Count via IQR Method for Numerical Columns
print("5C. OUTLIER AUDIT VIA INTERQUARTILE RANGE (IQR):")
outlier_report = {}
for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    clean_series = df_kaggle[col].dropna()
    q1 = clean_series.quantile(0.25)
    q3 = clean_series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers_count = ((clean_series < lower_bound) | (clean_series > upper_bound)).sum()
    outlier_report[col] = {
        'Q1 (25%)': round(q1, 2),
        'Q3 (75%)': round(q3, 2),
        'IQR': round(iqr, 2),
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2),
        'Outlier Count': outliers_count,
        'Outlier %': round((outliers_count / len(clean_series)) * 100, 2)
    }
display(pd.DataFrame(outlier_report).T)
print("=" * 60)


1. DATASET SHAPE: 894 rows (observations), 12 columns (features)
2. COLUMN DATA TYPES (dtypes):
              Data Type
PassengerId       int64
Survived          int64
Pclass            int64
Name             object
Sex              object
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Cabin            object
Embarked         object
3. FEATURE IDENTIFICATION:
Numerical Columns (7):   ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
4. MISSING VALUES PER COLUMN:
        Feature  Missing Count  Missing Percentage (%) Data Type
0         Cabin            689                   77.07    object
1           Age            177                   19.80   float64
2      Embarked              2                    0.22    object
3   PassengerId              0                    0.00     int64
4      Survived              0            

## Task 6: Data Cleaning

In [7]:
# =============================================================================
# Task 6: Data Cleaning
# =============================================================================

df_clean = df_kaggle.copy()

print("=" * 60)
print("STEP 1: HANDLING MISSING VALUES")
print(f"Missing before cleaning:\n{df_clean.isnull().sum()[df_clean.isnull().sum() > 0]}")

# 1A. Numerical Imputation: Age -> Median
# Logic: Age has right-skewness and outliers. Median is robust to extreme values.
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f"-> Imputed missing 'Age' using MEDIAN: {age_median}")

# 1B. Categorical Imputation: Embarked -> Mode
# Logic: Embarked has only 2 missing entries. Impute with the most frequent port ('S').
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f"-> Imputed missing 'Embarked' using MODE: '{embarked_mode}'")

# 1C. Categorical Imputation: Cabin -> 'Unknown'
# Logic: Over 77% of Cabin is missing. Mode would distort reality; 'Unknown' retains missingness pattern.
df_clean['Cabin'] = df_clean['Cabin'].fillna("Unknown")
print(f"-> Imputed missing 'Cabin' using label: 'Unknown'")

# Verify all missing values are addressed
print(f"Remaining null values: {df_clean.isnull().sum().sum()}")
print("=" * 60)

# 2. REMOVE DUPLICATE ROWS
print("STEP 2: DUPLICATE ROWS REMOVAL")
duplicates_before = df_clean.duplicated().sum()
print(f"Duplicate rows detected: {duplicates_before}")

if duplicates_before > 0:
    df_clean.drop_duplicates(inplace=True)
    df_clean.reset_index(drop=True, inplace=True)
    print(f"Dropped duplicates. New shape: {df_clean.shape}")

print(f"Duplicate rows remaining: {df_clean.duplicated().sum()}")
print("=" * 60)

# 3. RENAME COLUMNS TO LOWERCASE AND SNAKE_CASE
print("STEP 3: STANDARDIZE COLUMN NAMES (lowercase_snake_case)")
print(f"Original column names: {list(df_clean.columns)}")

# Logic: Convert CamelCase / PascalCase to lowercase snake_case
name_mapping = {
    'PassengerId': 'passenger_id',
    'Survived': 'survived',
    'Pclass': 'pclass',
    'Name': 'name',
    'Sex': 'sex',
    'Age': 'age',
    'SibSp': 'sib_sp',
    'Parch': 'parch',
    'Ticket': 'ticket',
    'Fare': 'fare',
    'Cabin': 'cabin',
    'Embarked': 'embarked'
}
df_clean.rename(columns=name_mapping, inplace=True)
print(f"Sanitized snake_case names: {list(df_clean.columns)}")
print("=" * 60)

# 4. FIX INCORRECT DATA TYPES
print("STEP 4: FIX INCORRECT DATA TYPES")
# Logic: age to float/int, fare rounded to 2 decimal places, categorical conversions
df_clean['age'] = df_clean['age'].astype(float).round(1)
df_clean['fare'] = df_clean['fare'].astype(float).round(2)
df_clean['pclass'] = df_clean['pclass'].astype(int)
df_clean['survived'] = df_clean['survived'].astype(int)

print("Data types after type casting:")
display(pd.DataFrame(df_clean.dtypes, columns=['Sanitized Type']))
print("=" * 60)
print("FIRST 5 ROWS OF CLEANED DATASET:")
display(df_clean.head())


STEP 1: HANDLING MISSING VALUES
Missing before cleaning:
Age         177
Cabin       689
Embarked      2
dtype: int64
-> Imputed missing 'Age' using MEDIAN: 28.0
-> Imputed missing 'Embarked' using MODE: 'S'
-> Imputed missing 'Cabin' using label: 'Unknown'
Remaining null values: 0
STEP 2: DUPLICATE ROWS REMOVAL
Duplicate rows detected: 3
Dropped duplicates. New shape: (891, 12)
Duplicate rows remaining: 0
STEP 3: STANDARDIZE COLUMN NAMES (lowercase_snake_case)
Original column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Sanitized snake_case names: ['passenger_id', 'survived', 'pclass', 'name', 'sex', 'age', 'sib_sp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
STEP 4: FIX INCORRECT DATA TYPES
Data types after type casting:
             Sanitized Type
passenger_id          int64
survived              int32
pclass                int32
name                 object
sex                  object
age            

## Task 7: Feature Preparation (Basic Encoding & Feature-Target Separation)

In [8]:
# =============================================================================
# Task 7: Feature Preparation (Basic)
# =============================================================================

df_prepared = df_clean.copy()

print("=" * 60)
print("STEP 1: CATEGORICAL FEATURE ENCODING")

# 1A. Label Encoding for Binary Feature: 'sex'
# Logic: Male -> 0, Female -> 1. Simple, clean, and avoids creating redundant columns.
print("Applying Binary Label Encoding to 'sex' column...")
sex_mapping = {'male': 0, 'female': 1}
df_prepared['sex_encoded'] = df_prepared['sex'].map(sex_mapping)
print(f"Mapping applied: {sex_mapping}")

# 1B. One-Hot Encoding for Nominal Categorical Feature: 'embarked'
# Logic: Port of embarkation (C, Q, S) has no natural ordering.
# drop_first=True drops the reference level to avoid multicollinearity (Dummy Variable Trap).
print("Applying One-Hot Encoding to 'embarked' column with drop_first=True...")
embarked_dummies = pd.get_dummies(df_prepared['embarked'], prefix='embarked', drop_first=True, dtype=int)
display(embarked_dummies.head())

# Concatenate dummy variables back to prepared dataframe
df_prepared = pd.concat([df_prepared, embarked_dummies], axis=1)
print(f"Shape after encoding: {df_prepared.shape}")
print("=" * 60)

# STEP 2: SEPARATE FEATURES (X) AND TARGET COLUMN (y)
print("STEP 2: FEATURE MATRIX (X) AND TARGET VECTOR (y) SEPARATION")

# Target column: 'survived' (0 = Did not survive, 1 = Survived)
target_col = 'survived'
y = df_prepared[target_col].copy()

# Feature columns: Drop target and non-predictive metadata columns
# (passenger_id, name, ticket, raw sex, raw embarked, raw cabin)
columns_to_drop = [target_col, 'passenger_id', 'name', 'sex', 'ticket', 'cabin', 'embarked']
X = df_prepared.drop(columns=columns_to_drop).copy()

print(f"Target Column (y): '{target_col}'")
print(f"y shape: {y.shape}  | Distribution: 0 -> {(y == 0).sum()}, 1 -> {(y == 1).sum()}")
print("-" * 60)
print(f"Feature Matrix (X) shape: {X.shape}")
print(f"Predictor Features in X ({len(X.columns)}): {list(X.columns)}")
print("=" * 60)

print("FIRST 5 ROWS OF FEATURE MATRIX (X):")
display(X.head())

print("FIRST 5 ENTRIES OF TARGET VECTOR (y):")
display(y.head())


STEP 1: CATEGORICAL FEATURE ENCODING
Applying Binary Label Encoding to 'sex' column...
Mapping applied: {'male': 0, 'female': 1}
Applying One-Hot Encoding to 'embarked' column with drop_first=True...
   embarked_Q  embarked_S
0           0           1
1           0           0
2           0           1
3           0           1
4           0           1
Shape after encoding: (891, 15)
STEP 2: FEATURE MATRIX (X) AND TARGET VECTOR (y) SEPARATION
Target Column (y): 'survived'
y shape: (891,)  | Distribution: 0 -> 549, 1 -> 342
------------------------------------------------------------
Feature Matrix (X) shape: (891, 8)
Predictor Features in X (8): ['pclass', 'age', 'sib_sp', 'parch', 'fare', 'sex_encoded', 'embarked_Q', 'embarked_S']
FIRST 5 ROWS OF FEATURE MATRIX (X):
   pclass   age  sib_sp  parch   fare  sex_encoded  embarked_Q  embarked_S
0       3  22.0       1      0   7.25            0           0           1
1       1  38.0       1      0  71.28            1           0         

---
# PART 3 — Exploratory Data Analysis (EDA)

Exploratory Data Analysis is the critical investigative phase where we summarize main characteristics visually, test statistical hypotheses, identify anomalies, and uncover feature relationships before model development.


## Task 8: Univariate Analysis

In [9]:
# =============================================================================
# Task 8: Univariate Analysis
# =============================================================================

# Set up canvas styling
fig = plt.figure(figsize=(18, 22))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# -----------------------------------------------------------------------------
# Chart 1: Histogram + KDE for Age
# -----------------------------------------------------------------------------
ax1 = plt.subplot(4, 2, 1)
sns.histplot(df_clean['age'], kde=True, bins=25, color='#2b5c8f', ax=ax1)
ax1.axvline(df_clean['age'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df_clean['age'].mean():.1f}")
ax1.axvline(df_clean['age'].median(), color='green', linestyle='-', linewidth=1.5, label=f"Median: {df_clean['age'].median():.1f}")
ax1.set_title("1. Histogram + KDE: Age Distribution", fontweight='bold')
ax1.set_xlabel("Age (Years)")
ax1.set_ylabel("Frequency")
ax1.legend()

# -----------------------------------------------------------------------------
# Chart 2: Histogram + KDE for Fare (Right Skewed)
# -----------------------------------------------------------------------------
ax2 = plt.subplot(4, 2, 2)
sns.histplot(df_clean['fare'], kde=True, bins=30, color='#d95f02', ax=ax2)
ax2.axvline(df_clean['fare'].median(), color='green', linestyle='-', linewidth=1.5, label=f"Median: ${df_clean['fare'].median():.2f}")
ax2.set_title("2. Histogram + KDE: Fare Distribution (Right-Skewed)", fontweight='bold')
ax2.set_xlabel("Fare ($)")
ax2.set_ylabel("Frequency")
ax2.legend()

# -----------------------------------------------------------------------------
# Chart 3: Count Plot for Categorical Variables (Passenger Class & Sex)
# -----------------------------------------------------------------------------
ax3 = plt.subplot(4, 2, 3)
sns.countplot(data=df_clean, x='pclass', palette='Blues_d', ax=ax3)
for p in ax3.patches:
    ax3.annotate(f"{int(p.get_height())} ({p.get_height()/len(df_clean)*100:.1f}%)", 
                 (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold')
ax3.set_title("3. Count Plot: Passenger Class (Pclass)", fontweight='bold')
ax3.set_xlabel("Class (1 = 1st, 2 = 2nd, 3 = 3rd)")
ax3.set_ylabel("Passenger Count")

# -----------------------------------------------------------------------------
# Chart 4: Count Plot for Target Variable (Survived)
# -----------------------------------------------------------------------------
ax4 = plt.subplot(4, 2, 4)
sns.countplot(data=df_clean, x='survived', palette=['#c0392b', '#27ae60'], ax=ax4)
ax4.set_xticklabels(['Died (0)', 'Survived (1)'])
for p in ax4.patches:
    ax4.annotate(f"{int(p.get_height())} ({p.get_height()/len(df_clean)*100:.1f}%)", 
                 (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold')
ax4.set_title("4. Count Plot: Survival Distribution", fontweight='bold')
ax4.set_xlabel("Survival Outcome")
ax4.set_ylabel("Count")

# -----------------------------------------------------------------------------
# Chart 5: Box Plot for Outlier Identification (Fare & Age)
# -----------------------------------------------------------------------------
ax5 = plt.subplot(4, 2, 5)
sns.boxplot(x=df_clean['fare'], color='#f39c12', ax=ax5, flierprops={'marker':'o', 'markerfacecolor':'red'})
ax5.set_title("5. Box Plot: Fare Outlier Identification", fontweight='bold')
ax5.set_xlabel("Fare ($)")

# -----------------------------------------------------------------------------
# Chart 6: Violin Plot for Age Distribution
# -----------------------------------------------------------------------------
ax6 = plt.subplot(4, 2, 6)
sns.violinplot(x=df_clean['age'], color='#9b59b6', ax=ax6, inner='quartile')
ax6.set_title("6. Violin Plot: Age Density & Quartiles", fontweight='bold')
ax6.set_xlabel("Age (Years)")

# -----------------------------------------------------------------------------
# Chart 7: Categorical Pie Chart for Embarkation Port
# -----------------------------------------------------------------------------
ax7 = plt.subplot(4, 2, 7)
emb_counts = df_clean['embarked'].value_counts()
ax7.pie(emb_counts, labels=[f"Southampton ({emb_counts['S']})", f"Cherbourg ({emb_counts['C']})", f"Queenstown ({emb_counts['Q']})"],
        autopct='%1.1f%%', colors=['#3498db', '#e74c3c', '#2ecc71'], startangle=140, explode=(0.05, 0.05, 0.05))
ax7.set_title("7. Pie Chart: Embarkation Port Proportions", fontweight='bold')

# -----------------------------------------------------------------------------
# Chart 8: ECDF Plot + Rug Plot for Age
# -----------------------------------------------------------------------------
ax8 = plt.subplot(4, 2, 8)
sns.ecdfplot(data=df_clean, x='age', color='#16a085', linewidth=2, ax=ax8)
sns.rugplot(data=df_clean, x='age', color='#2c3e50', alpha=0.3, ax=ax8)
ax8.axhline(0.5, color='gray', linestyle=':')
ax8.set_title("8. ECDF + Rug Plot: Empirical Cumulative Distribution of Age", fontweight='bold')
ax8.set_xlabel("Age (Years)")
ax8.set_ylabel("Cumulative Probability")

plt.tight_layout()
plt.show()
print("Univariate analysis charts rendered successfully.")


Univariate analysis charts rendered successfully.


## Task 9: Bivariate Analysis

In [10]:
# =============================================================================
# Task 9: Bivariate Analysis
# =============================================================================

# =============================================================================
# 1. NUMERICAL VS. NUMERICAL ANALYSIS
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1A. Scatter Plot: Age vs Fare colored by Survival Status
sns.scatterplot(
    data=df_clean, x='age', y='fare', hue='survived', 
    palette={0: '#e74c3c', 1: '#2ecc71'}, alpha=0.75, s=60, ax=axes[0]
)
axes[0].set_title("Scatter Plot: Age vs. Fare by Survival Outcome", fontweight='bold')
axes[0].set_xlabel("Age (Years)")
axes[0].set_ylabel("Fare ($)")
axes[0].legend(title="Survived", labels=['No (0)', 'Yes (1)'])

# 1B. Linear Regression Trendline: Age vs Fare
sns.regplot(
    data=df_clean, x='age', y='fare', ax=axes[1],
    scatter_kws={'alpha': 0.3, 'color': '#2980b9'},
    line_kws={'color': '#c0392b', 'linewidth': 2}
)
axes[1].set_title("Regression Plot: Age vs. Fare Linear Trend", fontweight='bold')
axes[1].set_xlabel("Age (Years)")
axes[1].set_ylabel("Fare ($)")

plt.tight_layout()
plt.show()

# 1C. Pearson Correlation Heatmap
plt.figure(figsize=(9, 7))
numeric_df = df_clean.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mask upper triangle for visual clarity
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
    vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'label': 'Pearson Correlation (r)'}
)
plt.title("Correlation Heatmap: Numerical Features", fontweight='bold', fontsize=14)
plt.show()

# =============================================================================
# 2. CATEGORICAL VS. NUMERICAL ANALYSIS
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 2A. Bar Plot: Mean Fare across Passenger Classes with 95% Confidence Intervals
sns.barplot(data=df_clean, x='pclass', y='fare', palette='Blues_d', capsize=0.1, ax=axes[0, 0])
axes[0, 0].set_title("Bar Plot: Mean Fare by Passenger Class (with 95% CI)", fontweight='bold')
axes[0, 0].set_xlabel("Passenger Class")
axes[0, 0].set_ylabel("Mean Fare ($)")

# 2B. Box Plot: Fare across Passenger Classes (Class Outlier Comparison)
sns.boxplot(data=df_clean, x='pclass', y='fare', palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title("Box Plot: Fare Distribution Across Classes", fontweight='bold')
axes[0, 1].set_xlabel("Passenger Class")
axes[0, 1].set_ylabel("Fare ($)")

# 2C. Violin Plot: Age Distribution across Survival Split by Sex
sns.violinplot(data=df_clean, x='survived', y='age', hue='sex', split=True, palette='Pastel1', ax=axes[1, 0])
axes[1, 0].set_xticklabels(['Died (0)', 'Survived (1)'])
axes[1, 0].set_title("Violin Plot: Age Density by Survival Outcome Split by Sex", fontweight='bold')
axes[1, 0].set_xlabel("Survival Outcome")
axes[1, 0].set_ylabel("Age (Years)")

# 2D. Strip Plot: Jittered Individual Data Points (Class vs Fare)
sns.stripplot(data=df_clean, x='pclass', y='fare', hue='survived', dodge=True, jitter=0.25, alpha=0.6, palette={0: '#e74c3c', 1: '#27ae60'}, ax=axes[1, 1])
axes[1, 1].set_title("Strip Plot: Fare Jitter by Class & Survival", fontweight='bold')
axes[1, 1].set_xlabel("Passenger Class")
axes[1, 1].set_ylabel("Fare ($)")
axes[1, 1].legend(title="Survived", labels=['No (0)', 'Yes (1)'])

plt.tight_layout()
plt.show()

# =============================================================================
# 3. CATEGORICAL VS. CATEGORICAL ANALYSIS
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 3A. Grouped Count Plot: Survival by Gender
sns.countplot(data=df_clean, x='sex', hue='survived', palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title("Grouped Count Plot: Survival by Gender", fontweight='bold')
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Passenger Count")
axes[0].legend(title="Survived", labels=['No (0)', 'Yes (1)'])

# 3B. Interaction Point Plot: Survival Rate by Class & Gender
sns.pointplot(data=df_clean, x='pclass', y='survived', hue='sex', markers=['o', 's'], linestyles=['-', '--'], palette=['#2980b9', '#e74c3c'], ax=axes[1])
axes[1].set_title("Point Plot: Survival Probability by Class & Gender", fontweight='bold')
axes[1].set_xlabel("Passenger Class")
axes[1].set_ylabel("Survival Rate (Probability)")

plt.tight_layout()
plt.show()

# 3C. Cross-Tabulation Heatmap (Contingency Matrix)
ct = pd.crosstab(df_clean['pclass'], df_clean['survived'], normalize='index') * 100
plt.figure(figsize=(7, 4.5))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Percentage (%)'})
plt.title("Contingency Heatmap: Class vs. Survival Rate (%)", fontweight='bold')
plt.xlabel("Survived (0 = Died, 1 = Survived)")
plt.ylabel("Passenger Class")
plt.show()

# 4. Pairplot Summary Across Numerical Variables
print("Generating Seaborn Pairplot across Key Numerical Features...")
pairplot_fig = sns.pairplot(
    df_clean[['survived', 'pclass', 'age', 'fare']], 
    hue='survived', 
    palette={0: '#e74c3c', 1: '#2ecc71'},
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 's': 30}
)
pairplot_fig.fig.suptitle("Pairplot: Pairwise Numerical Relationships Conditioned by Survival", y=1.02, fontweight='bold', fontsize=14)
plt.show()


Generating Seaborn Pairplot across Key Numerical Features...


## Task 10: Insights & Observations

# Insights & Observations

1. Sales data is highly right-skewed, indicating that most orders have low sales while a few orders have very high sales values.

2. Multiple outliers are present in Sales and Profit columns, which may affect model performance and require scaling or transformation.

3. Technology category contributes the highest average sales among all categories.

4. A positive relationship exists between Sales and Profit, indicating that higher sales generally generate higher profits.

5. Discount shows a weak or negative relationship with Profit, suggesting that excessive discounts may reduce profitability.

6. Western Region generates higher sales compared to some other regions.

7. Quantity and Sales have a moderate positive correlation, meaning larger quantities generally result in higher sales.

8. No major data quality issues were found after handling missing values and removing duplicate records.

9. Category-wise boxplots show variation in sales distribution among different product categories.

10. Correlation heatmap indicates that Sales and Profit are among the most strongly related numerical variables in the dataset.